# Determining the Sentiment of IMDB Reviews

In this notebook I will use the IMDb dataset to demonstrate how to build a sentiment analysis model. I'll preprocess the data, encode text, and train a neural network using TensorFlow and Keras.

### Step 1: Importing Required Libraries
Importing the necessary libraries for data manipulation, visualization, and model building.


In [16]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # 0=all, 1=INFO, 2=WARNING, 3=ERROR
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
tf.random.set_seed(42)

### Step 2: Loading the IMDb Dataset
Loading the dataset from a CSV file. The dataset contains movie reviews and their associated sentiment labels.

In [17]:
data=pd.read_excel('/data/IMDB_dataset.xlsx')
print(data.head())
print(data.shape)

                                              review sentiment
0  I thought this was a wonderful way to spend ti...  positive
1  Probably my all-time favorite movie, a story o...  positive
2  I sure would like to see a resurrection of a u...  positive
3  This show was an amazing, fresh & innovative i...  negative
4  Encouraged by the positive comments about this...  negative
(25000, 2)


### Step 3: Encode Sentiment Labels
Replacing the sentiment labels "negative" and "positive" with 0 and 1, respectively, for easier processing.

In [18]:
## .map() to replace all and store it to data['sentiment'].
## .fillna(0) for unmapped values.
## .astype(int) to define values as integer.
sentiment_map = {
    'positive': 1,
    'negative': 0
}
data['sentiment'] = data['sentiment'].map(sentiment_map).fillna(0).astype(int)
print(data['sentiment'].unique())

[1 0]


### Step 4: Define Target Variable
Extracting the target variable y which contains sentiment values.

In [19]:
y = data['sentiment']
print(y[0:10])

0    1
1    1
2    1
3    0
4    0
5    0
6    0
7    0
8    1
9    0
Name: sentiment, dtype: int64


In [20]:
print(data.shape)

(25000, 2)


### Step 5: Split Data into Train and Test Sets
Splitting the reviews and sentiments into training and testing sets with an 80-20 ratio.

In [21]:
X_train,X_test,y_train,y_test = train_test_split(data['review'],y,test_size=0.2,random_state=42)
print(f"""
Train samples: {X_train.shape[0]}
Test samples: {X_test.shape[0]}
"""
)


Train samples: 20000
Test samples: 5000



### Step 6: Check Sentiment Distribution
Ensuring the training data is balanced by checking the frequency of each sentiment.

In [22]:
frequency=y_train.value_counts()/y_train.shape[0]
print(frequency)

sentiment
1    0.50135
0    0.49865
Name: count, dtype: float64


### Step 7: Convert Targets to Dummy Vectors
Converting the target labels into one-hot encoded vectors for categorical classification.

In [23]:
### Assign the output to y_train and y_test
y_train = tf.keras.utils.to_categorical(y_train,num_classes=2)
y_test = tf.keras.utils.to_categorical(y_test,num_classes=2)
print(y_train.shape)
print(y_test.shape)

(20000, 2)
(5000, 2)


### Step 8: Text Vectorization (Multi-hot Encoding)
Setting `max_tokens` = 2412 to limit the vocabulary size for efficiency and creating a Keras `TextVectorization` layer with `output_mode`="multi_hot" and call adapt(`X_train`) to learn the vocabulary from the training data.

In [24]:
max_tokens = 2412
text_vectorization = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="multi_hot")

# Ensure X_train is a string Series for adaptation
# Re-splitting to guarantee a fresh string Series for adaptation
X_train_for_adapt, _, _, _ = train_test_split(data['review'], y, test_size=0.2, random_state=42)
text_vectorization.adapt(X_train_for_adapt)

In [25]:
print(y_train.shape)

(20000, 2)


In [26]:
X_train_vec = text_vectorization(X_train)
X_test_vec = text_vectorization(X_test)

print(X_train_vec.shape, X_test_vec.shape,y_train.shape, y_test.shape)

(20000, 2412) (5000, 2412) (20000, 2) (5000, 2)


### Step 9: Build the Neural Network Model
Building a simple Keras model for text classification with the following architecture:

- An input layer matching the shape of max_tokens.
- A dense layer with 32 units and ReLU activation.
- A Dropout layer with a dropout rate of 0.5.
- A final dense layer with 2 units and softmax activation for binary classification.

In [27]:
inputs = keras.Input(shape=(max_tokens, ))
x = keras.layers.Dense(32,activation = "relu")(inputs)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(2,activation = "softmax")(x)
model = keras.Model(inputs,outputs)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 2412)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │        77,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 77,282 (301.88 KB)

 Trainable params: 77,282 (301.88 KB)

 Non-trainable params: 0 (0.00 B)

### Step 10: Compile and Train the Model
Compiling the model using `adam` optimizer and train it for 5 epochs.

In [30]:
# Compile your model
model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

In [31]:
# Fit your model
model.fit(x=X_train_vec, y=y_train,
          epochs=5,
          batch_size=32)


Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8163 - loss: 0.3994
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8864 - loss: 0.2813
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9033 - loss: 0.2424
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9176 - loss: 0.2118
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9260 - loss: 0.1925


### Step 11: Evaluate the Model
Evaluating the model on the test set.

In [32]:
model.evaluate(x=X_test_vec, y=y_test)

157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8706 - loss: 0.3242


[0.32419002056121826, 0.8705999851226807]

### Conclusion and Next Steps
In this notebook, I have built successfully a baseline sentiment analysis model using IMDb movie reviews, achieving an accuracy of around 87% on the test set. While this is a strong starting point, there is room for improvement to create a more robust and accurate model:
- More complex architectures: more layers, neurons or regularization techniques
- Richer text representations: bigrams or pre-trained embeddings
- Hyperparameters tuning: number of epochs, batch size, and learning rate.